# 03 — REINVENT 4 generative design (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrdulasolutions/MCAS.Opensource/blob/main/notebooks/03_reinvent_generation_colab.ipynb)

> ⚠️ **Not medical advice.** Research/hypothesis use only.

What this notebook does:
1. Installs REINVENT 4 + downloads the public prior from Zenodo.
2. Seeds RL generation with quercetin, luteolin, and sulforaphane (our highest-priority anchors).
3. Runs a short reinforcement-learning stage with multi-objective scoring (QED + SA + Lipinski + scaffold similarity to seeds).
4. Writes generated SMILES to `outputs/reinvent_generated.csv`.

**Recommended runtime:** Colab GPU (T4 free tier is enough for a demo run). Mac CPU works but is very slow.

## Install (Colab)

REINVENT 4's install is sensitive to torch/CUDA versions. Use the pinned versions below.

In [ ]:
!pip -q install rdkit pandas numpy tqdm
# REINVENT 4 install (Colab):
# Option A — pip from MolecularAI's published wheel:
!pip -q install reinvent==4.6.* || echo 'reinvent pip wheel not found — falling back to git'
# Option B — git clone if pip fails (uncomment if needed):
# !git clone https://github.com/MolecularAI/REINVENT4.git && cd REINVENT4 && pip install -e .

In [ ]:
# Download the public REINVENT 4 priors from Zenodo (record IDs in the docs).
# Replace ZENODO_URL with the current public prior bundle.
ZENODO_URL = 'https://zenodo.org/records/12476471/files/reinvent4_priors.zip'  # update if version changes
import os, urllib.request, zipfile
os.makedirs('priors', exist_ok=True)
if not os.path.exists('priors/reinvent.prior'):
    print('Downloading priors...')
    urllib.request.urlretrieve(ZENODO_URL, 'priors/priors.zip')
    with zipfile.ZipFile('priors/priors.zip') as zf:
        zf.extractall('priors')
    print('Done.')

## Seeds

We seed RL with three anchors that span the rescue / maintenance / remission categories:
- **Quercetin** — flavonoid mast cell stabilizer (rescue / maintenance).
- **Luteolin** — BBB-crossing flavonoid (maintenance, neuro-MCAS).
- **Sulforaphane** — Nrf2 master activator (remission / upstream).

In [ ]:
SEEDS = {
    'Quercetin':    'O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12',
    'Luteolin':     'O=c1cc(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12',
    'Sulforaphane': 'CS(=O)CCCCN=C=S',
}
for n, s in SEEDS.items():
    print(n, s)

## Multi-objective scoring config

REINVENT 4 uses a TOML scoring config. Below is a minimal multi-objective template combining:
- QED (drug-likeness)
- SA score (synthesizability)
- Lipinski (rule-of-five)
- Tanimoto similarity to the seeds (keep close to the anchor scaffolds)

Save as `scoring.toml`.

In [ ]:
scoring_toml = '''
[parameters]
name = "mcas_seed_rl"

[[components]]
[components.QED]
[components.QED.endpoint]
name = "QED"
weight = 0.4

[[components]]
[components.SAScore]
[components.SAScore.endpoint]
name = "SA_score"
weight = 0.2

[[components]]
[components.TanimotoSimilarity]
[components.TanimotoSimilarity.endpoint]
name = "sim_to_seeds"
weight = 0.4
params.smiles = [
  "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12",
  "O=c1cc(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12",
  "CS(=O)CCCCN=C=S"
]
'''
with open('scoring.toml', 'w') as fh:
    fh.write(scoring_toml)
print('Wrote scoring.toml')

In [ ]:
# Run REINVENT 4 in 'staged_learning' (RL) mode. This is illustrative — exact CLI flags
# follow the upstream README (https://github.com/MolecularAI/REINVENT4).
import subprocess
cmd = [
    'reinvent',
    '--config', 'scoring.toml',
    '--mode', 'staged_learning',
    '--prior', 'priors/reinvent.prior',
    '--num-steps', '50',  # demo run; bump for production
    '--out-dir', 'outputs/reinvent_run',
]
print('Run:', ' '.join(cmd))
# subprocess.run(cmd, check=True)
print('(uncomment subprocess.run when REINVENT is installed)')

In [ ]:
# Parse REINVENT outputs into a tidy CSV for downstream docking + ranking.
import glob, pandas as pd, os
files = glob.glob('outputs/reinvent_run/*.csv')
if files:
    dfs = [pd.read_csv(f) for f in files]
    df = pd.concat(dfs, ignore_index=True)
    df.to_csv('outputs/reinvent_generated.csv', index=False)
    print('Wrote outputs/reinvent_generated.csv (', len(df), ' rows)')
else:
    print('No REINVENT outputs found yet.')

## Next

Drop `outputs/reinvent_generated.csv` into `04_virtual_screening.ipynb` to dock the novel molecules against MRGPRX2 / KIT / FcεRI, then `05_hypothesis_ranking.ipynb` to rank everything.